## Purpose
Understand all raw datasets before building cleaning and transformation pipelines.

## Scope
- Identify structure, grain, and key fields
- Surface data quality issues
- Define cleaning rules


# Helper Functions

In [183]:
import pandas as pd
from pathlib import Path
pd.set_option('display.max_columns', None)
import csv
import numpy

## get_sources
Obtain the sources of raw data used to analysis in this project, outputs a list of subfolders containing the name of the data source


In [184]:
def get_sources():
    """
    Return list of dataset folders in Data/RawData
    """
    current = Path.cwd()
    while not (current / "Data").exists():
        current = current.parent

    raw_path = current / "Data" / "RawData"
    
    return [
        folder.name for folder in raw_path.iterdir()
        if folder.is_dir()
    ]


## read_files
Obtain the raw files of sources used to analysis in this project, outputs a list of the names and extensions of the file

@param name of the source, obtained from get_sources function

In [185]:

def read_files(source):
    """
    Return list of readable file paths from Data/RawData/<source>
    """
    current = Path.cwd()
    while not (current / "Data").exists():
        current = current.parent

    data_path = current / "Data" / "RawData" / source
    
    return [
        file for file in data_path.iterdir()
        if file.suffix.lower() in [".csv", ".xlsx", ".xls"]
    ]

## pull_data
Reads the contents of the data, and outputs it as a dataframe

@param name of the filesource, outputted from read_files
outputs a dictionary of all the files from the source

In [186]:
def pull_data(source):
    """
    Load all readable files from Data/RawData/<source>.

    Parameters:
    - source (str): folder name under RawData (e.g., 'ward')

    Returns:
    - dict: {file_name_without_extension: DataFrame}
    """
    files = read_files(source)
    data = {}

    for file in files:
        file = Path(file)
        try:
            if file.suffix.lower() == ".csv":
                try:
                    df = pd.read_csv(file, on_bad_lines="skip", encoding="utf-8")
                except UnicodeDecodeError:
                    #print(f"⚠️ Encoding issue in {file.name}, retrying with latin1") #debugging
                    df = pd.read_csv(file, on_bad_lines="skip", encoding="latin1")
            elif file.suffix.lower() in [".xlsx", ".xls"]:
                df = pd.read_excel(file)
            else:
                continue
            data[file.stem] = df
            #print(f"Loaded: {file.name}")
        except Exception as e:
            print(f"\n❌ Error loading {file.name}")
            print(e)

            # 🔍 Debug: find problematic rows
            if file.suffix.lower() == ".csv":
                print("🔎 Scanning for bad rows...")

                with open(file, "r", encoding="utf-8") as f:
                    reader = csv.reader(f)
                    expected_len = len(next(reader))  # header length

                    for i, row in enumerate(reader, start=2):
                        if len(row) != expected_len:
                            print(f"\n⚠️ Bad row at line {i}:")
                            print(row)
                            break  # stop at first bad row
    return data

## full_summary or quick_summary
@Param data outputted by a specific dataset of the source

In [187]:
#Quick summary, or full summary, loads the dataframe and pulls the dtypes, % missing data, and # unique values
def quick_summary(df):
    return pd.DataFrame({
        "dtype": df.dtypes,
        "missing_pct": df.isnull().mean(),
        "n_unique": df.nunique()
    })

def full_summary(data):
    for name, df in data.items():
        print(f"\nSummary for: {name}")
        print(quick_summary(df))
    return

## compare_columns
@param data which contains the dictionary of all the data available for a source, outputted from pull_data

outputs a table showing which columns exist for a given dataset and which aren't from a union of all aggregatd columns for that source

In [199]:
def compare_columns(data):

    all_cols = sorted(set().union(*[df.columns for df in data.values()]))

    table = pd.DataFrame(index=all_cols)

    for name, df in data.items():
        table[name] = table.index.isin(df.columns)

    return table

# Dataset: 311 Service Requests - Customer Initiated
Index 0 on DataSources


## Description
The dataset contains information on customer initiated service requests received by 311 Toronto. 
This data was collected from multiple channels including telephone, fax, email, online self-serve requests, mobile API and Twitter.

311 manages City divisional service request data for Solid Waste Management, Transportation Services, Toronto Water, Municipal Licensing & Standards,
and Urban Forestry. For our purposes, we will be focusing on Toronto Water


## Source
City Open Data Catalogue - CKAN API 
https://open.toronto.ca/dataset/311-service-requests-customer-initiated/
Index # is 1 i 

## Files included
Readme
ServiceRequests csv files from 2010-2026

## Grain
One instance of a 311 service request

## Cleaning Rules

1. Remove non-data files (e.g., readme)

2. Standardize column names
   - convert to snake_case
   - lowercase, remove spaces/special characters

3. Align schema across datasets
   - ensure consistent column structure before combining

4. Handle missing values
   - assess missingness
   - decide to drop, impute, or retain

5. Remove duplicates
   - deduplicate based on primary key (e.g., request_id)

6. Enforce data types
   - convert date fields to datetime
   - ensure numeric fields are numeric
   - cast categorical fields appropriately

7. Standardize values
   - fix inconsistent categories (e.g., casing)
   - trim whitespace

8. Drop unnecessary columns
   - remove irrelevant or redundant fields

9. Validate data
   - ensure key fields are not null
   - remove invalid records


## Obtain source index and return summary of data

In [188]:
#Read available datasets available for data source
DataSources = get_sources()
print(DataSources)
source = (DataSources[0])
readable = read_files(source)


['311-service-requests-customer-initiated', 'current-and-future-climate', 'neighbourhoods', 'ward-profiles-25-ward-model', 'watermain-breaks', 'watermains']


In [189]:
data_311 = (pull_data(source))
print(data_311.keys())#returns the file names of the datasets in the pulled data as well as the index

dict_keys(['311-service-requests-readme', 'SR2010', 'SR2011', 'SR2012', 'SR2013', 'SR2014', 'SR2015', 'SR2016', 'SR2017', 'SR2018', 'SR2019', 'SR2020', 'SR2021', 'SR2022', 'SR2023', 'SR2024', 'SR2025', 'SR2026'])


In [190]:
#going forward we no longer need the read_me, we will duplicate the dictionary without it
updated_311 = {k: v for k, v in data_311.items() if k != '311-service-requests-readme'}
print(f"number of datasets from source: {len(updated_311)}")
print(f"names of remaining datasets: {updated_311.keys()}")

number of datasets from source: 17
names of remaining datasets: dict_keys(['SR2010', 'SR2011', 'SR2012', 'SR2013', 'SR2014', 'SR2015', 'SR2016', 'SR2017', 'SR2018', 'SR2019', 'SR2020', 'SR2021', 'SR2022', 'SR2023', 'SR2024', 'SR2025', 'SR2026'])


In [191]:
full_summary(updated_311)
quick_summary(updated_311['SR2010'])


,dtype,missing_pct,n_unique
Creation Date,object,0.000000,251595
Status,object,0.000000,6
First 3 Chars of Postal Code,object,0.000000,99
Intersection Street 1,object,0.929368,3114
Intersection Street 2,object,0.929533,2984
Ward,object,0.000000,44
Service Request Type,object,0.000000,369
Division,object,0.000000,7
Section,object,0.000000,12


In [201]:
#
compare_columns(updated_311)

,SR2010,SR2011,SR2012,SR2013,SR2014,SR2015,SR2016,SR2017,SR2018,SR2019,SR2020,SR2021,SR2022,SR2023,SR2024,SR2025,SR2026
Creation Date,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
Division,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
First 3 Chars of Postal Code,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
Intersection Street 1,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
Intersection Street 2,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
Section,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
Service Request Type,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
Status,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
Ward,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True


In [192]:
#view first
updated_311['SR2010'].head()
#check the unique values for a specific column


,Creation Date,Status,First 3 Chars of Postal Code,Intersection Street 1,Intersection Street 2,Ward,Service Request Type,Division,Section
0,2010-01-01 00:38:26.0000000,Closed,Intersection,High Park Blvd,Parkside Dr,Parkdale-High Park (13),Road - Sanding / Salting Required,Transportation Services,Road Operations
1,2010-01-01 01:19:18.0000000,Closed,M4T,NaN,NaN,Toronto Centre-Rosedale (27),Water Service Line-Turn On,Toronto Water,District Ops
2,2010-01-01 01:19:48.0000000,Cancelled,M1T,NaN,NaN,Scarborough-Agincourt (40),Water Service Line-Leaking,Toronto Water,District Ops
3,2010-01-01 01:24:18.0000000,Cancelled,M1T,NaN,NaN,Scarborough-Agincourt (40),Water Service Line-Leaking,Toronto Water,District Ops
4,2010-01-01 01:42:26.0000000,Closed,M1K,NaN,NaN,Scarborough Southwest (35),Watermain-Possible Break,Toronto Water,District Ops
